# MSH 3-T-junction array: logical gates, full Pfaffian method vs reduced method (dense reference)

The gates of `tjunctions_python_ed.ipynb` (√Z and Z from `exchange_12`, √X and X from `exchange_23` with `rot_Tx = 400`) as **logical
unitaries** on $|0\rangle_L = |00\rangle$, $|1\rangle_L = |11\rangle$, computed two ways from the same dense time evolution:

* **Full method** (CZ and period-finding notebooks, `signed_amplitudes`): the many-body amplitudes $\langle j|\hat U|k\rangle$ of the Gaussian
  (BCS) states, from the single-particle propagator $S_T$ on **all** $2N$ occupied quasiparticles, via the generalized Wick theorem
  (a Pfaffian). It gives signs and relative phases, and also the probability that the bulk stays unexcited.
* **Reduced method** (`logical_gates_tn.ipynb`): only the zero-mode block, the $4\times4$ real Majorana matrix
  $M_{ij} = \langle\gamma_i(0)|S_T\gamma_j(0)\rangle$ and its spin representation $\tilde U$. It needs only the two evolved mode vectors, so it
  also works with MPS (and sparse propagators), but it cannot see bulk excitations.

Then the reduced gates of the tensor-network evolution (`logical_gates_tn.csv`, from the TDVP runs of `msh_tjunctions_tn.ipynb`) are
compared with the dense ones.

**Code.** The model, the protocols and the zero-mode construction are imported from `tjunctions_model.py` (shared with
`tjunctions_python_ed.ipynb`); this notebook adds the Pfaffian, the amplitudes and the logical-gate algebra.

**Contents**
1. Setup: the fixed BdG unitary and the logical states
2. Time evolution of the full frame (Z and X gates)
3. Full method: sign-resolved amplitudes from a Pfaffian
4. Reduced method: Majorana matrix and logical unitary
5. Full vs reduced
6. Comparison with the tensor-network gates

**Runtime.** About 30 minutes for the evolution (section 2, single thread), cached in `logical_gates_python_ed_data/frames.npz`
(not in the repository); with the cache, a few seconds.

## 1. Setup: the fixed BdG unitary and the logical states

In [1]:
import os, time
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")     # single thread (before NumPy is imported), as in tjunctions_python_ed
import numpy as np
from scipy import linalg as la
from tqdm import trange
from tjunctions_model import *      # model, protocols, Hamiltonian, zero-mode construction, Gauss–Legendre step

DATA_DIR = "logical_gates_python_ed_data"
os.makedirs(DATA_DIR, exist_ok=True)
RUN_EVOLUTION = False               # True: recompute the frames of section 2 even if the cache exists

psi0, E0, mbs, mask = zero_mode_frame(H0)          # ψ_A, ψ_B in columns 2N, 2N+1; partners in 2N−1, 2N−2
states = [(0, 0), (0, 1), (1, 0), (1, 1)]
print(f"N = {N} sites, D = {4*N};  ‖psi0† psi0 − 1‖ = {la.norm(psi0.conj().T @ psi0 - np.eye(4*N)):.1e}")
print("energies of ψ_A, ψ_B:", np.real([np.vdot(psi0[:, 2*N+k], H0 @ psi0[:, 2*N+k]) for k in range(2)]))

N = 144 sites, D = 576;  ‖psi0† psi0 − 1‖ = 6.5e-14
energies of ψ_A, ψ_B: [1.09010706e-09 1.15785955e-09]


## 2. Time evolution of the full frame

The whole $4N\times4N$ frame is evolved with the sparse Gauss–Legendre `step` ($dt = 0.25$, columns renormalized after each step), exactly as in
`tjunctions_python_ed.ipynb`, and saved after one and after two exchanges. The single-particle propagator is then
$S_T = \psi(T)\,\psi_0^\dagger$ ($\psi_0$ is unitary).

In [2]:
p12, p23 = Exchange12(), Exchange23(rot_T=400.0)
T1 = {"Z": p12.duration(), "X": p23.duration()}
gate_f = {"Z": lambda t: f_array_12(p12, t if t < T1["Z"] else t - T1["Z"]),
          "X": lambda t: f_array_23(p23, t if t < T1["X"] else t - T1["X"])}

def evolve_frame(ft, psi, t0, t1, dt=DT):
    "Full-frame evolution t0 → t1 with the Gauss–Legendre step (as `evolve` in tjunctions_python_ed, without readout)."
    nsteps = int(round((t1 - t0) / dt))
    for k in trange(nsteps, mininterval=30):
        psi = step(psi, t0 + k * dt, dt, ft)
        psi = psi / np.linalg.norm(psi, axis=0)
    return psi

frames_file = os.path.join(DATA_DIR, "frames.npz")
if RUN_EVOLUTION or not os.path.isfile(frames_file):
    frames, secs = {}, {}
    for kind in ("Z", "X"):
        t_ = time.perf_counter()
        frames[kind + "1"] = evolve_frame(gate_f[kind], psi0.copy(), 0.0, T1[kind])
        frames[kind + "2"] = evolve_frame(gate_f[kind], frames[kind + "1"], T1[kind], 2 * T1[kind])
        secs[kind] = time.perf_counter() - t_
        print(f"{kind}: T1 = {T1[kind]:.0f}, two exchanges in {secs[kind]:.0f} s")
    np.savez(frames_file, psi0=psi0, **frames, secs_Z=secs["Z"], secs_X=secs["X"])
d = np.load(frames_file)
# use the frame the cache was evolved from (eigh's phases of degenerate bulk states depend on the BLAS build and threads)
psi0 = d["psi0"]
Ec = np.real(np.einsum("ij,ik,kj->j", psi0.conj(), H0, psi0))
assert la.norm(H0 @ psi0 - psi0 * Ec) < 1e-8 and np.allclose(psi0[:, 2*N:2*N+2], zero_mode_frame(H0)[0][:, 2*N:2*N+2], atol=1e-8), \
    "cached psi0 is not the zero-mode frame of H0: set RUN_EVOLUTION = True"
S = {key: d[key] @ psi0.conj().T for key in ("Z1", "Z2", "X1", "X2")}       # S_T after one / two exchanges
for key, ST in S.items():
    print(f"{key}: ‖S_T† S_T − 1‖ = {la.norm(ST.conj().T @ ST - np.eye(4*N)):.1e}")
print(f"evolution time (cached run): Z {float(d['secs_Z']):.0f} s, X {float(d['secs_X']):.0f} s")

Z1: ‖S_T† S_T − 1‖ = 3.7e-12
Z2: ‖S_T† S_T − 1‖ = 6.4e-12
X1: ‖S_T† S_T − 1‖ = 6.7e-12
X2: ‖S_T† S_T − 1‖ = 1.3e-11
evolution time (cached run): Z 692 s, X 1363 s


## 3. Full method: sign-resolved amplitudes from a Pfaffian

The construction of the CZ notebook (`signed_amplitudes`, here without mid-protocol projections; the period-finding version is
`projected_amplitudes` in `period_finding_signed_amplitudes.py`):

* **States.** $|k\rangle$ for zero-mode occupations $k = (n_A, n_B)$ is the Slater/BCS state with the columns `mask_of(k)` of $\psi_0$ occupied
  (as `logical_masks`). With $\phi_m$ = column $2N+m$ and $b(w) = w^\dagger\Psi$, $|k\rangle = a^\dagger_{m_1}a^\dagger_{m_2}\cdots|00\rangle$ ($m_1 < m_2$),
  where the creation operator of mode $m$ is represented by $C\phi_m$ and its adjoint by $\phi_m$.
* **Common reference.** Both $|j\rangle$ and $|k\rangle$ are written relative to a reference $|r\rangle$:
  $\langle j|\hat U|k\rangle / \langle r|\hat U|r\rangle = \langle r|X(r)\,X^\dagger(j)\,\hat U\,X(k)\,X^\dagger(r)|r\rangle$, and the operators left of $\hat U$ are moved
  through it with $\hat U^\dagger b(v)\hat U = b(S_T^\dagger v)$.
* **Generalized Wick theorem.** The normalized expectation value of the operator string is the Pfaffian of the antisymmetric matrix of
  pairwise contractions $x_i^\dagger(1-\rho)Cx_j$, with the *mixed* density $1 - \rho = 1 - W_rG^{-1}W_L^\dagger$, $W_L = S_T^\dagger W_r$,
  $G = W_L^\dagger W_r$, $W_r$ the occupied columns of $|r\rangle$. This is where the whole occupied frame and its evolution enter.
* **Normalization.** $|\langle r|\hat U|r\rangle| = |\det G|^{1/2}$; its sign is not fixed but common to all amplitudes, so moduli and relative
  phases are meaningful and the global phase is not. The reference is the candidate with the largest $|\det G|$.

`pfaffian` is the Parlett–Reid elimination with pivoting; `Cvec` is the particle–hole conjugation of the kwant spinor ($C = \tau_y\sigma_y K$).

In [3]:
def pfaffian(A):
    """Pfaffian of a complex antisymmetric matrix (Parlett–Reid with pivoting)."""
    A = np.array(A, dtype=complex)
    n = A.shape[0]
    if n % 2:
        return 0j
    pf = 1 + 0j
    for k in range(0, n - 1, 2):
        kp = k + 1 + np.argmax(np.abs(A[k + 1:, k]))
        if kp != k + 1:
            A[[k + 1, kp], :] = A[[kp, k + 1], :]
            A[:, [k + 1, kp]] = A[:, [kp, k + 1]]
            pf = -pf
        if A[k + 1, k] == 0:
            return 0j
        pf *= A[k, k + 1]
        if k + 2 < n:
            tau = A[k, k + 2:] / A[k, k + 1]
            A[k + 2:, k + 2:] += np.outer(tau, A[k + 2:, k + 1]) - np.outer(A[k + 2:, k + 1], tau)
    return pf

Cvec = Cop                                   # particle–hole conjugate of one BdG vector (tjunctions_model)

def mask_of(bits):
    """Occupied columns of the state with zero-mode occupations `bits`; partners in reverse order, excited states empty."""
    m = np.zeros(4 * N, bool)
    m[2 * N:2 * N + len(bits)] = bits
    m[:2 * N] = np.flip(~m[2 * N:])
    return m

def signed_amplitudes(pairs, S_T, psi0, candidates=states):
    """⟨j|Û|k⟩ for (j, k) in pairs, up to one common sign; also returns the reference r and |⟨r|Û|r⟩|."""
    nq = len(pairs[0][0])
    phi = [psi0[:, 2 * N + m] for m in range(nq)]
    Cphi = [Cvec(v) for v in phi]
    SdT = S_T.conj().T
    X = lambda bits: [Cphi[m] for m in range(nq) if bits[m]]              # a†_{m1} a†_{m2} …
    Xd = lambda bits: [phi[m] for m in reversed(range(nq)) if bits[m]]    # its adjoint
    logdets = [np.linalg.slogdet(psi0[:, mask_of(c)].conj().T @ S_T @ psi0[:, mask_of(c)])[1] for c in candidates]
    r = candidates[int(np.argmax(logdets))]
    Wr = psi0[:, mask_of(r)]
    WL = SdT @ Wr
    G = WL.conj().T @ Wr
    norm_r = np.exp(0.5 * np.linalg.slogdet(G)[1])
    out = {}
    for jb, kb in pairs:
        ops = [SdT @ v for v in X(r) + Xd(jb)] + X(kb) + Xd(r)
        if not ops:
            out[(jb, kb)] = norm_r + 0j
            continue
        P = np.column_stack(ops); CP = np.column_stack([Cvec(v) for v in ops])
        Mx = P.conj().T @ CP - (P.conj().T @ Wr) @ np.linalg.solve(G, WL.conj().T @ CP)   # x_i†(1 − ρ) C x_j
        Cm = np.triu(Mx, 1)
        out[(jb, kb)] = pfaffian(Cm - Cm.T) * norm_r
    return out, r, norm_r

# checks: Pf² = det, and S_T = 1 gives ⟨j|k⟩ = δ_jk
A = np.random.default_rng(1).normal(size=(8, 8)) + 1j * np.random.default_rng(2).normal(size=(8, 8)); A = A - A.T
print(f"|Pf(A)² − det A| = {abs(pfaffian(A)**2 - np.linalg.det(A)):.1e}")
amp, r, nr = signed_amplitudes([(j, k) for j in states for k in states], np.eye(4 * N, dtype=complex), psi0)
print(f"identity propagator: max|A_jk − δ_jk| = {max(abs(amp[(j, k)] - (j == k)) for j in states for k in states):.1e}")

full = {}
for key, ST in S.items():
    amp, r, nr = signed_amplitudes([(j, k) for j in states for k in states], ST, psi0)
    full[key] = dict(A=np.array([[amp[(j, k)] for k in states] for j in states]), r=r, norm_r=nr)
    print(f"{key}: reference r = {r}, |⟨r|Û|r⟩| = {nr:.6f}")

|Pf(A)² − det A| = 3.7e-11
identity propagator: max|A_jk − δ_jk| = 3.9e-15
Z1: reference r = (0, 1), |⟨r|Û|r⟩| = 0.999999
Z2: reference r = (0, 1), |⟨r|Û|r⟩| = 0.999999
X1: reference r = (0, 1), |⟨r|Û|r⟩| = 0.691313
X2: reference r = (0, 0), |⟨r|Û|r⟩| = 0.044149


## 4. Reduced method: Majorana matrix and logical unitary

The Majoranas of each fermion mode follow from the frame, $\gamma_{2k-1} = (\psi_k + C\psi_k)/\sqrt2$ and $\gamma_{2k} = (\psi_k - C\psi_k)/(i\sqrt2)$, the
same convention as $\psi_k = (\gamma_{2k-1} + i\gamma_{2k})/\sqrt2$ in the tensor-network notebook. $M_{ij} = \langle\gamma_i|S_T\gamma_j\rangle$ is real
(its imaginary part is printed as a check). `jw_majoranas`, `logical_unitary` and `logical_even_block` are the functions of
`logical_gates_tn.ipynb` (section 2 there explains them, including the Jordan–Wigner convention: $f_m$ annihilates the positive-energy mode
$\psi_m$, so $n_m = 1$ means $\psi_m$ occupied, as in `mask_of`), in Python.

In [4]:
sq2 = np.sqrt(2)
gamma0 = np.column_stack([v for k in range(2) for v in ((psi0[:, 2*N+k] + Cvec(psi0[:, 2*N+k])) / sq2,
                                                        (psi0[:, 2*N+k] - Cvec(psi0[:, 2*N+k])) / (1j * sq2))])
print(f"Majoranas: ‖γ†γ − 1‖ = {la.norm(gamma0.conj().T @ gamma0 - np.eye(4)):.1e}, "
      f"max ‖Cγ − γ‖ = {max(la.norm(Cvec(g) - g) for g in gamma0.T):.1e}, centres x =",
      np.round([np.real(np.vdot(g, np.repeat(xs, 4) * g)) for g in gamma0.T], 1))

def braid_matrix(S_T):
    Mc = gamma0.conj().T @ S_T @ gamma0
    return Mc.real, np.abs(Mc.imag).max()

def jw_majoranas():
    a = np.array([[0, 1], [0, 0]], complex); Zp = np.diag([1.0, -1.0]).astype(complex); I2 = np.eye(2)
    fA, fB = np.kron(a, I2), np.kron(Zp, a)
    # f_m annihilates the +E mode ψ_m = (γ_{2m−1} + iγ_{2m})/√2 (b(w) = w†Ψ is antilinear): γ̂_{2m} = +i(f_m − f_m†)
    return [fA + fA.conj().T, 1j * (fA - fA.conj().T), fB + fB.conj().T, 1j * (fB - fB.conj().T)]

def logical_unitary(M):
    """Spin representation Ũ (order 00, 01, 10, 11) of the Majorana rotation M, and its residual."""
    U_, s_, Vt = la.svd(M); Mo = U_ @ Vt                     # nearest orthogonal matrix
    assert la.det(Mo) > 0, "M has det −1: not a parity-preserving rotation"
    g = jw_majoranas(); best = None
    A = la.logm(Mo).real; A = (A - A.T) / 2
    for sgn in (1, -1):
        Uu = la.expm(sum(sgn * A[i, j] * g[i] @ g[j] for i in range(4) for j in range(4)) / 4)
        res = max(la.norm(Uu @ g[j] @ Uu.conj().T - sum(g[i] * Mo[i, j] for i in range(4))) for j in range(4))
        if best is None or res < best[1]:
            best = (Uu, res)
    return best

sx, sy, sz = (np.array(m, complex) for m in ([[0, 1], [1, 0]], [[0, -1j], [1j, 0]], [[1, 0], [0, -1]]))
IDEAL = {"√X": la.expm(-1j * np.pi / 4 * sx), "√X†": la.expm(1j * np.pi / 4 * sx), "X": sx,
         "√Z": np.diag([1, 1j]), "√Z†": np.diag([1, -1j]), "Z": sz}

def logical_even_block(U):
    """Even block on {|00⟩, |11⟩}: θ, n̂, P(00→11) and the overlaps |Tr(V†U_L)|/2 with the ideal gates."""
    UL = U[np.ix_([0, 3], [0, 3])]
    W = UL / np.sqrt(la.det(UL))
    th = 2 * np.arccos(np.clip(np.real(np.trace(W)) / 2, -1, 1))
    n = np.zeros(3) if np.sin(th / 2) < 1e-12 else np.array([np.real(1j * np.trace(s @ W)) / (2 * np.sin(th / 2)) for s in (sx, sy, sz)])
    return dict(UL=UL, theta=np.degrees(th), n=n, P01=abs(UL[1, 0])**2,
                F={k: abs(np.trace(V.conj().T @ UL)) / 2 for k, V in IDEAL.items()})

reduced = {}
for key, ST in S.items():
    M, imM = braid_matrix(ST)
    U, res = logical_unitary(M)
    reduced[key] = dict(M=M, U=U, L=logical_even_block(U), imM=imM, res=res, MtM=la.norm(M.T @ M - np.eye(4)))
    print(f"{key}: max|Im M| = {imM:.1e}, ‖MᵀM − 1‖ = {reduced[key]['MtM']:.1e}, residual {res:.1e}")

Majoranas: ‖γ†γ − 1‖ = 2.3e-15, max ‖Cγ − γ‖ = 0.0e+00, centres x = [-46.6 -17.5  16.5  45.6]
Z1: max|Im M| = 1.6e-15, ‖MᵀM − 1‖ = 1.4e-06, residual 6.4e-16
Z2: max|Im M| = 1.4e-15, ‖MᵀM − 1‖ = 2.8e-06, residual 1.7e-15
X1: max|Im M| = 1.8e-15, ‖MᵀM − 1‖ = 7.0e-05, residual 2.4e-15
X2: max|Im M| = 2.3e-15, ‖MᵀM − 1‖ = 1.0e-04, residual 1.6e-15


## 5. Full vs reduced

Per gate and stage: the even-sector transition probabilities $|\langle j|\hat U|k\rangle|^2$ from the full method, the probability of staying
unexcited ($\sum_j|\langle j|\hat U|k\rangle|^2$ over the logical outputs, per input $k$), the full probabilities renormalized to the logical subspace,
and the reduced ones; then the relative phases, and the whole even block entry by entry: the full block with each column divided by the
square root of its unexcited probability, against the reduced $\tilde U$, after removing the global phase (which the full method does not fix).
With the Jordan–Wigner convention of section 4 the two agree to the bulk-excitation level; with the opposite sign of $\hat\gamma_{2m}$ the
diagonal and off-diagonal phases come out conjugated.

In [5]:
names = {"Z1": "√Z (one exchange_12)", "Z2": "Z (two exchange_12)", "X1": "√X (one exchange_23)", "X2": "X (two exchange_23)"}
ev = [0, 3]
rel = lambda U: np.degrees(np.angle(U[3, 3] / U[0, 0]))
for key in S:
    Af, Ur = full[key]["A"], reduced[key]["U"]
    Pf = np.abs(Af[np.ix_(ev, ev)])**2; stay = Pf.sum(axis=0); Pr = np.abs(Ur[np.ix_(ev, ev)])**2
    L = reduced[key]["L"]
    print(f"── {names[key]}")
    print(f"   full    |⟨out|Û|in⟩|² (rows 00, 11; columns in = 00, 11): {np.round(Pf, 6).tolist()}")
    print(f"   unexcited probability per input: {np.round(stay, 6).tolist()}")
    print(f"   full, renormalized: {np.round(Pf / stay, 6).tolist()}")
    print(f"   reduced:            {np.round(Pr, 6).tolist()}   max difference {np.abs(Pf / stay - Pr).max():.1e}")
    print(f"   arg⟨11|Û|11⟩ − arg⟨00|Û|00⟩: full {rel(Af):+.4f}°, reduced {rel(Ur):+.4f}°")
    if abs(Ur[3, 0]) > 1e-6:
        print(f"   arg⟨11|Û|00⟩ − arg⟨00|Û|00⟩: full {np.degrees(np.angle(Af[3, 0] / Af[0, 0])):+.4f}°, "
              f"reduced {np.degrees(np.angle(Ur[3, 0] / Ur[0, 0])):+.4f}°")
    i, j = np.unravel_index(np.argmax(np.abs(Af[np.ix_(ev, ev)])), (2, 2))
    unphase = lambda B: B * np.exp(-1j * np.angle(B[i, j]))
    Bf, Br = Af[np.ix_(ev, ev)] / np.sqrt(stay)[None, :], Ur[np.ix_(ev, ev)]
    print(f"   even block entry by entry: max|U_full/√P_stay − Ũ| = {np.abs(unphase(Bf) - unphase(Br)).max():.1e}")
    print(f"   reduced gate: θ = {L['theta']:.4f}°, n̂ = {np.round(L['n'], 4).tolist()}, "
          + ", ".join(f"{k} {v:.6f}" for k, v in L["F"].items()))

── √Z (one exchange_12)
   full    |⟨out|Û|in⟩|² (rows 00, 11; columns in = 00, 11): [[0.999999, 0.0], [0.0, 0.999998]]
   unexcited probability per input: [0.999999, 0.999998]
   full, renormalized: [[1.0, 0.0], [0.0, 1.0]]
   reduced:            [[1.0, 0.0], [0.0, 1.0]]   max difference 3.5e-29
   arg⟨11|Û|11⟩ − arg⟨00|Û|00⟩: full -90.7180°, reduced -90.7180°
   even block entry by entry: max|U_full/√P_stay − Ũ| = 3.3e-12
   reduced gate: θ = 90.7180°, n̂ = [0.0, 0.0, -1.0], √X 0.496858, √X† 0.496858, X 0.000000, √Z 0.006265, √Z† 0.999980, Z 0.711523
── Z (two exchange_12)
   full    |⟨out|Û|in⟩|² (rows 00, 11; columns in = 00, 11): [[0.999997, 0.0], [0.0, 0.999997]]
   unexcited probability per input: [0.999997, 0.999997]
   full, renormalized: [[1.0, 0.0], [0.0, 1.0]]
   reduced:            [[1.0, 0.0], [0.0, 1.0]]   max difference 2.2e-28
   arg⟨11|Û|11⟩ − arg⟨00|Û|00⟩: full +178.5641°, reduced +178.5641°
   even block entry by entry: max|U_full/√P_stay − Ũ| = 1.0e-11
   reduced g

## 6. Comparison with the tensor-network gates

`logical_gates_tn.csv` holds $M$ from the TDVP evolution of $\psi_A$, $\psi_B$ (`logical_gates_tn.ipynb`). The Majoranas of the two codes are
localized and paired the same way, but each $\gamma$ is fixed only up to its sign, so the comparison uses quantities that do not depend on
that convention: $|M_{ij}|$, the leakage, $P(00\to11)$ and the rotation angle $\theta$ of the logical gate.

In [6]:
import csv
tn = {}
with open("logical_gates_tn.csv") as fh:
    for row in csv.DictReader(fh):
        M = np.array([[float(row[f"M{i}{j}"]) for j in range(1, 5)] for i in range(1, 5)])
        tn[row["gate"] + row["stage"]] = dict(M=M, leak=(float(row["leak_A"]), float(row["leak_B"])))
print(f"{'gate':22s} {'max||M_TN|−|M_ED||':>19s} {'P(00→11) TN':>12s} {'ED':>9s} {'θ TN':>10s} {'θ ED':>10s} "
      f"{'leak TN (A, B)':>20s} {'leak ED (A, B)':>20s}")
for key in S:
    Mt, Me = tn[key]["M"], reduced[key]["M"]
    Lt, Le = logical_even_block(logical_unitary(Mt)[0]), reduced[key]["L"]
    leak_ed = [1 - np.sum(np.abs(gamma0.conj().T @ (S[key] @ psi0[:, 2*N+k]))**2) for k in range(2)]
    print(f"{names[key]:22s} {np.abs(np.abs(Mt) - np.abs(Me)).max():19.1e} {Lt['P01']:12.6f} {Le['P01']:9.6f} "
          f"{Lt['theta']:9.4f}° {Le['theta']:9.4f}°  ({tn[key]['leak'][0]:.2e}, {tn[key]['leak'][1]:.2e})  "
          f"({leak_ed[0]:.2e}, {leak_ed[1]:.2e})")

gate                    max||M_TN|−|M_ED||  P(00→11) TN        ED       θ TN       θ ED       leak TN (A, B)       leak ED (A, B)
√Z (one exchange_12)               4.9e-09     0.000000  0.000000   90.7180°   90.7180°  (9.72e-07, 1.65e-11)  (9.74e-07, -5.33e-15)
Z (two exchange_12)                8.5e-09     0.000000  0.000000  178.5641°  178.5641°  (1.99e-06, 2.07e-11)  (2.00e-06, -5.55e-15)
√X (one exchange_23)               1.6e-08     0.522078  0.522078   92.5308°   92.5308°  (3.49e-05, 1.23e-07)  (3.49e-05, 1.24e-07)
X (two exchange_23)                1.8e-08     0.998051  0.998051  174.9391°  174.9391°  (3.42e-05, 3.49e-05)  (3.42e-05, 3.49e-05)
